In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
#ω_a = 28.25396662345805
ω_a = 25.338776456203686/4

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
#ω_a = 3.85*(rand(1)[1]/5+0.9)*2*pi
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 1.2
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
sin_pump =  - 2*E_J*sin(ϵ_0)
Δ_a = sqrt(Z_a)/(sqrt(2)*ϕ_quantum*ω_a)*sin_pump
Δ_b = sqrt(Z_b)/(sqrt(2)*ϕ_quantum*ω_b)*sin_pump

dephasing = (2*Δ_a)/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (2*Δ_b)/sqrt(2) * sqrt(Z_b)/ϕ_quantum

In [ ]:
dephasing = -4*E_J*sin(ϵ_0)*(φ_a^2 + φ_b^2/2)/ω_a

In [ ]:
E_J = 1/((φ_a^2 + φ_b^2/2)/ω_a)/4 * 2*pi

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.5pi
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2;

In [ ]:
int_pi = -2*E_J/pi*sin(ϵ_0)*(φ_a^2 + φ_b^2/2)/ω_a

In [ ]:
t_max=30/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)

ϵ_d=10*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list)

        Final_mode[(re_i,im_j)] = a_rot
    end
end

In [ ]:
p0 = plot()

    # Overlay scatter points
for (re_i, im_j) in keys(Final_mode)
    scatter!(p0,
            real.(Final_mode[(re_i, im_j)]),
            imag.(Final_mode[(re_i, im_j)]),
            ms=1, label="", color=:black)
end

# Add labels and title
plot!(p0,
    xlabel=L"\Re(a)",
    ylabel=L"\Im(a)",
    title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
    top_margin=5mm)

# Comparing with the Stark-shift model

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_p*cos(ω_p*t))
    coupling = (cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum)) / ϕ_quantum / sqrt(2)
    
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max=100/κ_b
a_0 = 0.0
b_0 = 0.0

ω_d_eff = 2.017*2*pi#ω_b

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);
F, freqs = FFT_perso(real.(b), t_list);

In [ ]:
plot(freqs, abs.(F), linewidth=.1)
#vline!([ω_d_eff/2/pi], label=L"ω_{d,eff}/2pi", ls=:dash)
vline!([2.017], ls=:dash)
plot!(xlims=(1,3), yaxis=:log)

In [ ]:
F, freqs = FFT_perso(real.(a), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([ω_a/2/pi], label=L"ω_a/2pi")
vline!([ω_d_eff/2/pi/2], label=L"ω_{d,eff}/2pi")
plot!(xlims=(0,5.5), yaxis=:log)

In [ ]:
mask = (freqs .> 0.2) .& (freqs .< 5)

freqs_m = freqs[mask]
F_m     = F[mask]

freq_max = abs(freqs_m[argmax(abs.(F_m))])
a_rot = (a.-sum(a)/length(a)).*exp.(1im*freq_max*2*pi*t_list);

In [ ]:
plot(real.(a_rot[1:end].+real(sum(a)/length(a))), imag.(a_rot[1:end]).+imag(sum(a)/length(a)))

In [ ]:
t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(t_save);

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)

ϵ_d=10*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list);

        Final_mode[(re_i,im_j)] = a_rot
    end
end

In [ ]:
p0 = plot()

    # Overlay scatter points
for (re_i, im_j) in keys(Final_mode)
    scatter!(p0,
            real.(Final_mode[(re_i, im_j)]),
            imag.(Final_mode[(re_i, im_j)]),
            ms=1, label="", color=:black)
end

# Add labels and title
plot!(p0,
    xlabel=L"\Re(a)",
    ylabel=L"\Im(a)",
    title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
    top_margin=5mm)